# 05 — Full CNN + MLP + Fusion + LSTM Model
The complete architecture: per-timestep CNN + Sensor MLP + Feature Fusion, fed through an LSTM to capture temporal trends, ending in three risk heads (Stress / Water / Pest). Finishes by generating a spatial risk map.

In [ ]:
import sys, os
sys.path.append(os.path.abspath("../src"))

import numpy as np
import torch
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

from dataset import SequenceCropDataset
from models import CNNLSTMRiskModel, MultiTaskRiskLoss
from inference import generate_field_risk_maps, plot_risk_maps
from preprocessing import load_sentinel2_scene

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

PROCESSED_DIR = "../data/processed"
FIELD_ID = sorted(os.listdir(PROCESSED_DIR))[0]
SEQ_DIR = os.path.join(PROCESSED_DIR, FIELD_ID, "sequences")


## Load pre-built sequences from `02_preprocessing.ipynb`

In [ ]:
image_seqs = np.load(os.path.join(SEQ_DIR, "image_seqs.npy"))
sensor_seqs = np.load(os.path.join(SEQ_DIR, "sensor_seqs.npy"))
labels = np.load(os.path.join(SEQ_DIR, "labels.npy"))
splits = np.load(os.path.join(SEQ_DIR, "splits.npz"))
train_idx, val_idx, test_idx = splits["train"], splits["val"], splits["test"]

print("image_seqs:", image_seqs.shape, "sensor_seqs:", sensor_seqs.shape, "labels:", labels.shape)


## Datasets & loaders

In [ ]:
train_ds = SequenceCropDataset(image_seqs[train_idx], sensor_seqs[train_idx], labels[train_idx])
val_ds = SequenceCropDataset(image_seqs[val_idx], sensor_seqs[val_idx], labels[val_idx])
test_ds = SequenceCropDataset(image_seqs[test_idx], sensor_seqs[test_idx], labels[test_idx])

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=8, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=8, shuffle=False)

print(f"train: {len(train_ds)}  val: {len(val_ds)}  test: {len(test_ds)}")


## Model, loss, optimizer

In [ ]:
in_channels = image_seqs.shape[2]
sensor_features = sensor_seqs.shape[2]

model = CNNLSTMRiskModel(
    in_channels=in_channels,
    sensor_features=sensor_features,
    img_dim=128, sensor_dim=32, fused_dim=128,
    lstm_hidden=64, lstm_layers=1,
).to(device)

criterion = MultiTaskRiskLoss(weights=(1.0, 1.0, 1.0))
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)


## Training loop (with per-head loss tracking)

In [ ]:
def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss = 0.0
    head_losses = {"stress": 0.0, "water": 0.0, "pest": 0.0}
    with torch.set_grad_enabled(train):
        for batch in loader:
            imgs = batch["image_seq"].to(device)
            sensors = batch["sensor_seq"].to(device)
            label = batch["label"].to(device)

            preds = model(imgs, sensors)
            loss, parts = criterion(preds, label)

            if train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
                optimizer.step()

            bs = imgs.size(0)
            total_loss += loss.item() * bs
            for k in head_losses:
                head_losses[k] += parts[k] * bs

    n = len(loader.dataset)
    return total_loss / n, {k: v / n for k, v in head_losses.items()}


N_EPOCHS = 25
history = {"train_loss": [], "val_loss": []}
best_val_loss = float("inf")

for epoch in range(1, N_EPOCHS + 1):
    train_loss, train_parts = run_epoch(train_loader, train=True)
    val_loss, val_parts = run_epoch(val_loader, train=False)
    scheduler.step(val_loss)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    print(f"epoch {epoch:02d}  train={train_loss:.4f} (s{train_parts['stress']:.3f}/"
          f"w{train_parts['water']:.3f}/p{train_parts['pest']:.3f})  "
          f"val={val_loss:.4f} (s{val_parts['stress']:.3f}/w{val_parts['water']:.3f}/p{val_parts['pest']:.3f})")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        os.makedirs("../data/processed/checkpoints", exist_ok=True)
        torch.save(
            {"model_state_dict": model.state_dict(),
             "in_channels": in_channels, "sensor_features": sensor_features},
            "../data/processed/checkpoints/cnn_lstm_best.pt",
        )


## Loss curves

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history["train_loss"], label="train")
plt.plot(history["val_loss"], label="val")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend()
plt.title("CNN + MLP + Fusion + LSTM — training curve")
plt.show()


## Test-set evaluation

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

model.eval()
all_preds, all_targets = [], []
with torch.no_grad():
    for batch in test_loader:
        imgs = batch["image_seq"].to(device)
        sensors = batch["sensor_seq"].to(device)
        out = model(imgs, sensors)
        pred = torch.stack([out["stress_risk"], out["water_risk"], out["pest_risk"]], dim=1)
        all_preds.append(pred.cpu().numpy())
        all_targets.append(batch["label"].numpy())

all_preds = np.concatenate(all_preds)
all_targets = np.concatenate(all_targets)

for i, name in enumerate(["stress_risk", "water_risk", "pest_risk"]):
    mae = mean_absolute_error(all_targets[:, i], all_preds[:, i])
    r2 = r2_score(all_targets[:, i], all_preds[:, i])
    print(f"{name:12s} MAE={mae:.4f}  R2={r2:.4f}")


## Generate a spatial risk map for the field
Stitches per-patch predictions back into full-scene rasters using saved patch coordinates.

In [ ]:
from dataset import load_processed_field

dates, patches_by_date, sensors, coords = load_processed_field(FIELD_ID, PROCESSED_DIR)

# rebuild a sequence per current patch location using the last SEQ_LEN dates (must match training seq_len)
SEQ_LEN = image_seqs.shape[1]
recent_dates = range(len(dates) - SEQ_LEN, len(dates))
n_locations = patches_by_date[0].shape[0]

field_image_seqs = np.stack(
    [np.stack([patches_by_date[t][loc] for t in recent_dates]) for loc in range(n_locations)]
).astype(np.float32)  # (N_locations, T, C, H, W)

sensor_seq_full = np.stack([sensors[t] for t in recent_dates]) if sensors is not None else \
    np.zeros((SEQ_LEN, sensor_features), dtype=np.float32)
field_sensor_seqs = np.tile(sensor_seq_full, (n_locations, 1, 1)).astype(np.float32)

scene, _ = load_sentinel2_scene(sorted(
    __import__("glob").glob(f"../data/raw/sentinel2/{FIELD_ID}/*.tif")
)[-1])
scene_shape = scene.shape[1:]  # (H, W)

risk_maps = generate_field_risk_maps(
    model, field_image_seqs, field_sensor_seqs, coords, scene_shape,
    patch_size=64, device=device,
)
fig = plot_risk_maps(risk_maps)
plt.show()


### Next steps
- Swap the field-broadcast labels for real per-location ground truth (scouting points, yield-loss maps, etc.) once available — this is the single highest-leverage improvement.
- Try a bidirectional LSTM or a Transformer encoder in place of the LSTM if sequences get longer.
- Export `risk_maps` as GeoTIFFs (reuse the `profile` from `load_sentinel2_scene`) to view in QGIS.
- Wire the trained checkpoint into `app/app.py` for a simple risk-map serving demo.